In [ ]:
from wsi.train import Trainer
from wsi.nets.vision_transformer import *
from wsi.nets.dino import *
from wsi.utils import get_logger,img_show
import torch
import torchvision as tv
from tqdm import tqdm
import torch_geometric.nn as nn_g
from torch_geometric.utils import from_networkx
from skimage.io import imread
from sklearn.model_selection import train_test_split,KFold
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score
import cv2 as cv
import networkx as nx


In [ ]:
class AttModule(nn.Module):
    def __init__(self, num_classes, init=True, input_feat_dim=384):
        super(AttModule, self).__init__()
        self.input_feat_dim = input_feat_dim
        self.classifier = nn.Linear(self.input_feat_dim, num_classes)
        
        self.D = 512
        self.K = 1
        self.attention = nn.Sequential(
            nn.Linear(self.input_feat_dim, self.D),
            nn.Tanh(),
            nn.Linear(self.D, self.K)
        )
        if init:
            self._initialize_weights()

    def forward(self, x):
        # Attention module
        A_ = self.attention(x)  # NxK
        A_ = torch.transpose(A_, 1, 0)  # KxN
        A = F.softmax(A_, dim=1)  # softmax over N

        x = torch.mm(A, x)  # KxL
        x = self.classifier(x)
        return x, A_

    def _initialize_weights(self):
        for y, m in enumerate(self.modules()):
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                for i in range(m.out_channels):
                    m.weight.data[i].normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.01)
                m.bias.data.zero_()

class AttModule_GNN(nn.Module):
    def __init__(self, num_classes, init=True, input_feat_dim=384):
        super(AttModule_GNN, self).__init__()
        
        self.input_feat_dim = input_feat_dim
        self.classifier = nn.Linear(self.input_feat_dim, num_classes)
        
        self.D = 512
        self.K = 1

        self.attention1 = nn_g.SAGEConv(self.input_feat_dim,self.D)
        self.attention2 = nn_g.SAGEConv(self.D,self.K)
        
        if init:
            self._initialize_weights()

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Attention module
        A = self.attention1(x,edge_index)
        A = F.relu(A)
        A_ = self.attention2(A,edge_index) # N,1
        A = F.relu(A_)
        A = F.softmax(A_, dim=1)  # N,1
        x = A.T@x  # KxL
        x = self.classifier(x)
        return x, A_

    def _initialize_weights(self):
        for y, m in enumerate(self.modules()):
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                for i in range(m.out_channels):
                    m.weight.data[i].normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.01)
                m.bias.data.zero_()



class MIL(nn.Module):
    def __init__(self, encoder, aggregator, device="cuda:0"):
        super().__init__()
        self.device = torch.device(device)

        self.encoder = encoder.to(self.device)
        self.aggregator = aggregator.to(self.device)

    # 获取各实例的特征
    def get_instance_feature(self, patches, bsize=16):
        features = []
        for i in range(0,len(patches),bsize):
            f = torch.concat(patches[i:i+bsize],dim=0).float().to(self.device)
            features.append(self.encoder(f))
        features = torch.concat(features,0)
        return features

    # 聚合所有实例特征获得包级特征
    def get_aggregation(self,features):
        return self.aggregator(features)
    
    def forward(self,instance_list):
        features = self.get_instance_feature(instance_list)
        features = self.get_aggregation(features)
        return features


class MIL_GNN(nn.Module):
    def __init__(self, encoder, aggregator, device="cuda:0"):
        super().__init__()
        self.device = torch.device(device)

        self.encoder = encoder.to(self.device)
        self.aggregator = aggregator.to(self.device)
        
    def get_instance_feature(self, patches, bsize=16):
        patches = patches.float()
        features = []
        for i in range(0,len(patches),bsize):
            f = patches[i:i+bsize]
            features.append(self.encoder(f))
        features = torch.concat(features,0)
        return features
    
    # 聚合所有实例特征获得包级特征
    def get_aggregation(self,features):
        return self.aggregator(features)

    def forward(self,G):
        instance_list = G.img.squeeze(1)
        features = self.get_instance_feature(instance_list)
        G.x = features
        features = self.get_aggregation(G)
        return features

In [ ]:
path_list = []
for path,dirs,files in os.walk("../../Breast Cancer Cell Segmentation_datasets/Images/"):
    if len(dirs)==0:
        for f in files:
            if f.endswith("tif"):
                path_list.append(os.path.join(path,f))
data = []
for f in path_list:
    img = imread(f)
    img_t = cv.cvtColor(img, cv.COLOR_RGB2BGR)
    hsv = cv.cvtColor(img_t, cv.COLOR_BGR2HSV)
    lower_hsv = np.array([100, 10, 46])
    upper_hsv = np.array([160, 255, 250])
    mask = cv.inRange(hsv, lower_hsv, upper_hsv)
    mask[mask>0]=1
    H,W = mask.shape
    w = 64
    s = 32
    img_list = []
    yx_list = []
    for y in range(0,H-s,s):
        for x in range(0,W-s,s):
            if (mask[y:y+w,x:x+w].sum()/(w**2))>0.75:
                m = torch.tensor(img[y:y+w,x:x+w].copy()).permute(2,0,1).unsqueeze(0)
                img_list.append(m)
                yx_list.append([y,x])

    label = torch.tensor([0 if "benign" in f else 1])

    edge_list = []
    for i in range(len(yx_list)):
        for j in range(i):
            (y_i,x_i),(y_j,x_j) = yx_list[i],yx_list[j]
            if abs(y_i-y_j)<=s and abs(x_i-x_j)<=s:
                edge_list.append([i,j])
    G = nx.Graph()
    for i,(img,(y,x)) in enumerate(zip(img_list,yx_list)):
        G.add_node(i,img=img,h=y,w=x)
    G.add_edges_from(edge_list)

    data.append([G,label])
train_set,val_set = train_test_split(data,test_size=0.2,random_state=42)
len(train_set),len(val_set)

In [ ]:
img = imread(f)
img_t = cv.cvtColor(img, cv.COLOR_RGB2BGR)
hsv = cv.cvtColor(img_t, cv.COLOR_BGR2HSV)
lower_hsv = np.array([100, 10, 46])
upper_hsv = np.array([160, 255, 250])
mask = cv.inRange(hsv, lower_hsv, upper_hsv)
mask[mask>0]=1
H,W = mask.shape
w = 64
s = 32
img_list = []
yx_list = []
for y in range(0,H-s,s):
    for x in range(0,W-s,s):
        if (mask[y:y+w,x:x+w].sum()/(w**2))>0.75:
            m = torch.tensor(img[y:y+w,x:x+w].copy()).permute(2,0,1).unsqueeze(0)
            img_list.append(m)
            yx_list.append([y,x])
label = torch.tensor([0 if "benign" in f else 1])

edge_list = []
for i in range(len(yx_list)):
    for j in range(i):
        (y_i,x_i),(y_j,x_j) = yx_list[i],yx_list[j]
        if abs(y_i-y_j)<=s and abs(x_i-x_j)<=s:
            edge_list.append([i,j])
G = nx.Graph()
for i,(y,x) in enumerate(yx_list):
    G.add_node(i,h=y,w=x)
G.add_edges_from(edge_list)

data.append([img_list,label,G])


In [ ]:
class MILTrainer(Trainer):
    def __init__(self, model, optim, loss_func, save_path, device, scheduler=None):
        super().__init__(model, optim, loss_func, save_path, device, scheduler)

    def train_one_epoch(self, train_loader, fp16_scaler=None):
        self.model.train()
        loss_list = []
        for G,label in tqdm(train_loader):
            imgs = [G.nodes[n]["img"].to(self.device) for n in G]
            label = label.to(self.device)

            bag_pred,_ = self.model(imgs)
            bag_pred = torch.softmax(bag_pred,1)
            loss_teacher = self.loss_func(bag_pred,label)
            self.optim.zero_grad()
            loss_teacher.backward()
            self.optim.step()
            loss_list.append(loss_teacher.item())
        return np.mean(loss_list)
    
    def val_one_epoch(self, val_loader):
        self.model.eval()
        loss_list = []
        for G,label in tqdm(val_loader):
            imgs = [G.nodes[n]["img"].to(self.device) for n in G]
            label = label.to(self.device)

            bag_pred,_ = self.model(imgs)
            bag_pred = torch.softmax(bag_pred,1)
            loss_teacher = self.loss_func(bag_pred,label)
            loss_list.append(loss_teacher.item())
        return np.mean(loss_list)

class GNN_MILTrainer(Trainer):
    def __init__(self, model, optim, loss_func, save_path, device, scheduler=None):
        super().__init__(model, optim, loss_func, save_path, device, scheduler)

    def train_one_epoch(self, train_loader, fp16_scaler=None):
        self.model.train()
        loss_list = []
        for G,label in tqdm(train_loader):
            G = from_networkx(G).to(self.device)
            label = label.to(self.device)

            bag_pred,_ = self.model(G)
            bag_pred = torch.softmax(bag_pred,1)
            loss_teacher = self.loss_func(bag_pred,label)
            self.optim.zero_grad()
            loss_teacher.backward()
            self.optim.step()
            loss_list.append(loss_teacher.item())
        return np.mean(loss_list)
    
    def val_one_epoch(self, val_loader):
        self.model.eval()
        loss_list = []
        for G,label in tqdm(val_loader):
            G = from_networkx(G).to(self.device)
            label = label.to(self.device)

            bag_pred,_ = self.model(G)
            bag_pred = torch.softmax(bag_pred,1)
            loss_teacher = self.loss_func(bag_pred,label)
            loss_list.append(loss_teacher.item())
        return np.mean(loss_list)
    

In [ ]:
device = torch.device("cuda:0")
encoder = tv.models.resnet50(weights=tv.models.ResNet50_Weights.IMAGENET1K_V2)
encoder.fc = nn.Identity() # 最终输出1,2048
for p in encoder.parameters():
    p.requires_grad = False
# aggregator = AttModule(2,input_feat_dim=2048).cuda()
# mil = MIL(encoder,aggregator)
aggregator = AttModule_GNN(2,input_feat_dim=2048).to(device)
mil = MIL_GNN(encoder,aggregator)

# 优化器
optim_aggregator = torch.optim.Adam(filter(lambda p : p.requires_grad, mil.parameters()), lr=3e-4)

# train
epoches = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim_aggregator,T_max=50,eta_min=1e-5)
# trainer = MILTrainer(mil,optim_aggregator,nn.CrossEntropyLoss(),"./model/temp/",device,scheduler=scheduler)
trainer = GNN_MILTrainer(mil,optim_aggregator,nn.CrossEntropyLoss(),"./model/temp/",device,scheduler=scheduler)
trainer.train(train_set,val_set,epoches,5)


In [ ]:
# model = torch.load("./model/temp/model.pth")
# label_list = []
# pred_list = []
# for G,label in tqdm(val_set):
#     imgs = [G.nodes[n]["img"].to(device) for n in G]
#     bag_pred,_ = model(imgs)
#     bag_pred = torch.argmax(bag_pred,1)
#     pred_list.append(bag_pred.item())
#     label_list.append(label.item())

# print([label_list,pred_list,accuracy_score(label_list,pred_list), precision_score(label_list,pred_list), recall_score(label_list,pred_list), f1_score(label_list,pred_list), roc_auc_score(label_list,pred_list)])

model = torch.load("./model/temp/model.pth")
label_list = []
pred_list = []
for G,label in tqdm(val_set):
    G = from_networkx(G).to(device)
    bag_pred,_ = model(G)
    bag_pred = torch.argmax(bag_pred,1)
    pred_list.append(bag_pred.item())
    label_list.append(label.item())

print([label_list,pred_list,accuracy_score(label_list,pred_list), precision_score(label_list,pred_list), recall_score(label_list,pred_list), f1_score(label_list,pred_list), roc_auc_score(label_list,pred_list)])

In [ ]:
# f_id = 30
# f = path_list[f_id]
# G,label = data[f_id]
# imgs = [G.nodes[n]["img"].to(device) for n in G]
# bag_pred,A = model(imgs)
# A = F.softmax(A)
# A = A.squeeze(0).detach().cpu().numpy()

# img = imread(f)
# img_t = cv.cvtColor(img, cv.COLOR_RGB2BGR)
# hsv = cv.cvtColor(img_t, cv.COLOR_BGR2HSV)
# lower_hsv = np.array([100, 10, 46])
# upper_hsv = np.array([160, 255, 250])
# mask = cv.inRange(hsv, lower_hsv, upper_hsv)
# mask[mask>0]=1

# H,W = mask.shape
# w = 64
# s = 32
# att = np.zeros_like(mask,dtype=np.float32)
# count = 0
# for y in range(0,H-s,s):
#     for x in range(0,W-s,s):
#         if (mask[y:y+w,x:x+w].sum()/(w**2))>0.75:
#             att[y:y+w,x:x+w]=A[count]
#             count+=1
# att = (att-att.min())/(att.max()-att.min())

# img_show(att)
# img_show(img)
# img_show(np.concat([img/255,np.expand_dims(att,-1)],-1))
# f,bag_pred.argmax(1),label